In [ ]:
# Exercício 7 - Implementação do classificador com fronteira linear e quadrática

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score
from sklearn.datasets import make_circles
import seaborn as sns

# Gerar dados
x, y = make_circles(n_samples=1000, noise=0.1, factor=0.2, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=23)

# Adicionar bias (1), x1 e x2
X_train = np.c_[np.ones((len(x_train),1)), x_train]
X_test = np.c_[np.ones((len(x_test),1)), x_test]

# One-hot encoding
def to_one_hot(y):
    n_classes = y.max() + 1
    m = len(y)
    Y_one_hot = np.zeros((m, n_classes))
    Y_one_hot[np.arange(m), y] = 1
    return Y_one_hot

y_train_oh = to_one_hot(y_train)
y_test_oh = to_one_hot(y_test)

# Softmax
def softmax(logits):
    exps = np.exp(logits)
    return exps / np.sum(exps, axis=1, keepdims=True)

# Função de erro
def error_function(x, a, y, epsilon=1e-7):
    logits = x.dot(a)
    y_prob = softmax(logits)
    return -np.mean(np.sum(y * np.log(y_prob + epsilon), axis=1))

# Classificador
def classifier(x, a):
    logits = x.dot(a)
    y_prob = softmax(logits)
    return np.argmax(y_prob, axis=1).reshape(-1,1)

# Treinamento
alpha = 0.5
n_iterations = 3000
a = np.random.randn(X_train.shape[1], y_train_oh.shape[1])

Jgd = np.zeros(n_iterations+1)
Jgd_v = np.zeros(n_iterations+1)

Jgd[0] = error_function(X_train, a, y_train_oh)
Jgd_v[0] = error_function(X_test, a, y_test_oh)

for iteration in range(n_iterations):
    logits = X_train.dot(a)
    y_prob = softmax(logits)
    gradients = (1/len(y_train)) * X_train.T.dot(y_prob - y_train_oh)
    a = a - alpha * gradients
    Jgd[iteration+1] = error_function(X_train, a, y_train_oh)
    Jgd_v[iteration+1] = error_function(X_test, a, y_test_oh)

# Plot Erro
plt.figure(figsize=(8,5))
plt.plot(Jgd, label="Treino")
plt.plot(Jgd_v, label="Validação")
plt.xlabel('Épocas')
plt.ylabel('Erro')
plt.title('Erro vs Épocas')
plt.legend()
plt.grid()
plt.show()

# Matriz de confusão
y_pred_class = classifier(X_test, a)
mat = confusion_matrix(y_test, y_pred_class)

plt.figure(figsize=(5,5))
sns.heatmap(mat, annot=True, fmt='d', cmap="Blues")
plt.title('Matriz de Confusão')
plt.xlabel('Verdadeiro')
plt.ylabel('Predito')
plt.show()

# Curva ROC
y_prob = softmax(X_test.dot(a))[:,1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label='Curva ROC (área = {:.2f})'.format(roc_auc))
plt.plot([0,1],[0,1],'r--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.grid()
plt.show()

# Relatório de classificação
print(classification_report(y_test, y_pred_class))

# Acurácia
print('Acurácia:', accuracy_score(y_test, y_pred_class))
